In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=0)

agent = create_agent(
    # "gpt-5-nano",
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='645d0120-a90a-4b0a-ba42-55e8d66b1907'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'95242c19-a914-4004-9ed4-d2f9911280bc': 'EjQKMgERTTIPzE/4LHEmbBTDwLPCylOuUJDTGILVb7ykApfLfp2yrYyV3Lvj7r+6EOj3cpSN'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4130-3ec5-77a0-8175-9082c1807c57-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '95242c19-a914-4004-9ed4-d2f9911280bc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 22, 'total_tokens': 91, 'input_token_details': {'cache_read': 0}}),
 

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='69d4840d-427b-48e9-b11c-e0065028fe51'),
              AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today?", 'extras': {'signature': 'EjQKMgERTTIPxqSHwf2GBFnjrCwm03/fofl177KIKzlIYM5Lz5NPDawA1BhhSGrtEX9tAeCo'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4131-24ca-7e93-8ce5-9a5c85ded101-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    # "gpt-5-nano",
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='83cdacdb-9abd-4023-ad46-aab4fc3ba7ba'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'a62f4496-70a3-4c9d-8ac7-76c11e5d895a': 'EjQKMgERTTIPglPSON4n/1FXXYAstvTT9aOp7/dxEzVsvNilgU7oBlnTR+iqvd9jg+GzWajA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4131-9ea4-7000-b3fd-18161d2a6ca4-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'a62f4496-70a3-4c9d-8ac7-76c11e5d895a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_token_details': {'cache_read': 0}}),


In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='83cdacdb-9abd-4023-ad46-aab4fc3ba7ba'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'a62f4496-70a3-4c9d-8ac7-76c11e5d895a': 'EjQKMgERTTIPglPSON4n/1FXXYAstvTT9aOp7/dxEzVsvNilgU7oBlnTR+iqvd9jg+GzWajA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4131-9ea4-7000-b3fd-18161d2a6ca4-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'a62f4496-70a3-4c9d-8ac7-76c11e5d895a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_token_details': {'cache_read': 0}}),
